# Stratospheric Polar Vortex Diagnostics: An analysis with Numeric Weather Predictions
This notebook demonstrates the application of the arctic Python package to analyze the geometry and dynamics of the stratospheric polar vortex (SPV). It applies the methodology of Hannachi et al. (2011) to diagnostics derived from the Icosahedral Nonhydrostatic (ICON) general circulation model with upper atmosphere (UA-ICON). The aim is to replicate and evaluate SPV regime classification using elliptic indices and assess the model's ability to capture extreme vortex events.

# Table of Contents
- [Introduction](#introduction)
- [Data](#data)
- [Methodology](#methodology)
    - [Scaling](#scaling)
    - [Seasonality](#seasonality)
        - [Durbin Watson Test](#durbin-watson-test)
        - [Autocorrelation Function](#autocorrelation-function)
        - [Single Spectrum Analysis](#singular-spectrum-analysis)
        - [Extended Empirical Orthogonal Function](#extended-empirical-orthogonal-function)
    - [Gap statistic](#gap-statistic)
    - [Hierarchical clustering](#hierarchical-clustering)
- [Results](#results)
    - [Statistical Profiles of Clusters](#statistical-profiles-of-clusters)
    - [Physical Interpretation and Literature Comparison](#physical-interpretation-and-literature-comparison)
- [Appendix](#appendix)
    - [Partial Autocorrelation](#computation-of-the-partial-autocorrelation-function)
    - [Year over Year Averages](#year-over-year-averages)
    - [Abbreviations](#abbreviations)

# Introduction
The stratospheric polar vortex (SPV) plays a central role in modulating mid-latitude weather patterns, particularly during winter. Sudden stratospheric warming (SSW) events, which can split or displace the vortex, are of great interest due to their impacts on the troposphere.

This study builds upon the methodology of Hannachi et al. (2011, hereafter H11), who used geometric moments derived from potential vorticity fields to characterize vortex morphology. Here, we apply similar techniques to the extracted features from the numerical weather prediction (NWP) using the Icosahedral Nonhydrostatic (ICON) general circulation model with upper atmosphere (UA-ICON). We aim to replicate and assess H11's clustering results using elliptic indices. Further, the results provide an impression about the models capacity to simulate extreme vortex events.

# Data
The data used in this study were provided and preprocessed by the Institute of Atmospheric Physics (IAP) in Kühlungsborn. The data includes daily diagnostics from the NWP of the UA-ICON model, derived using the IDL-based ELDI and SSW packages. Specifically, the dataset contains the daily elliptic indices of the stratospheric polar vortex at 10hPa geopotential height and major warming diagnostics.

The data are split into three files:

- `*_cen.csv`: Contains event-centered diagnostics, including central SSW dates, i.e. when the wind reversed from westerly to easterly, persistence, maximum eastward wind, intensity, and accumulated intensity.
- `*_msw.csv`: Flags daily major warming events with corresponding zonally averaged wind speed.
- `*_d.csv`: Includes daily geometric vortex diagnostics: area, centroid latitude/longitude (latcent, loncent), aspect ratio (ar), orientation angle (theta), and kurtosis. This file also marks split events (S) and includes diagnostics for identified sub-vortices, along with wave diagnostics at 60° latitude.

The dataset spans 1979–2040 and includes the same indices used in the full analysis demonstration replicating the results of Hannachi et al. (2011) on ERA5 reanalysis data. This analysis will focus on the following geometric moments: area, aspect ratio (ar), centroid latitude (latcent), and kurtosis.

In [ ]:
# import vortexclust in general
import vortexclust

# import other important libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Import specific functions from vortexclust
from vortexclust.io.loader import read_data
from vortexclust.io.cleaner import no_white_space, to_date

# Read data
nwp_d = read_data("../data/SWXClust/NWP4A60/i4a60e2_d.csv")
nwp_cen = read_data("../data/SWXClust/NWP4A60/i4a60m2t-cen.csv")
nwp_msw = read_data("../data/SWXClust/NWP4A60/i4a60m2t-msw.csv")

# Remove white space from column names
no_white_space(nwp_d)
no_white_space(nwp_cen)
no_white_space(nwp_msw)

# Change string to datetime for better comparison
to_date(nwp_d, 'string', format='%d.%m.%Y-%H:%M:%S')
to_date(nwp_cen, 'string', format='%d.%m.%Y-%H:%M')
to_date(nwp_msw, 'string', format='%d.%m.%Y-%H:%M')

# Convert columns to numeric
col_convert = [
    'area1', 'obj_area1', 'latcent1', 'loncent1', 'theta1', 'ar1',
    'area2', 'obj_area2', 'latcent2', 'loncent2', 'theta2', 'ar2'
]
nwp_d[col_convert] = nwp_d[col_convert].apply(pd.to_numeric, errors='coerce')
nwp_d = nwp_d.fillna(0)

# Merge nwp data to one big dataframe
nwp_all = nwp_d.merge(nwp_msw, on ='string', how='left', suffixes=[None, '_msw']).\
    merge(nwp_cen, on='string', how='left', suffixes = ['_d', '_cen'])

# Handle categorical string encodings
from sklearn.preprocessing import LabelEncoder
le_mw = LabelEncoder()
nwp_all['MW'] = le_mw.fit_transform(nwp_all.MW)
print(f"Transformed 'MW':\n{le_mw.inverse_transform([1])} to 1\n{le_mw.inverse_transform([0])} to 0")
le_form = LabelEncoder()
nwp_all['form'] = le_form.fit_transform(nwp_all.form)
print(f"Transformed 'form': \n {le_form.inverse_transform([1])} to 1\n{le_form.inverse_transform([0])} to 0")

# Fill remaining NaNs in central diagnostics
nwp_all[nwp_cen.columns[-4:]] = nwp_all[nwp_cen.columns[-4:]].fillna(0)

# Drop constant and redundant columns
nwp_all.drop(['D_d', 'level', 'hour',
               'D_cen', 'num', 'counter_cen', 'time_cen',
               'D_msw', 'counter_msw', 'time_msw'
               ], axis=1, inplace=True)
# Sort by date
nwp_all = nwp_all.sort_values('string').reset_index(drop=True)

# check
nwp_all.info()

# Methodology

In H11, a winter period (December to March, DJFM) is selected, and data range from 1958 to 2002. The authors note activity peaks in December - January and an oscillation period around 120-130d. As mentioned before, the NWP data range from 1979 to 2040. The diagrams below depict the consecutive days of DJFM. For better orientation, a second axis with the actual dates is added. However, since only winter months are shown, the timeline skips directly from March to December.

## Scaling
Normalization refers broadly to any transformation of data making them more compatible (Gewers et al., 2021). Common approaches include *MinMax* scaling, which maps each feature to a given range $\left[min, max\right]$, and *Standard* scaling, which transforms each variable to have a mean of zero and variance of one.

$$x' = \frac{x-\mu}{\sigma}$$

H11 does not explicitly state the applied scaling method. However, based on the timeseries diagrams in their study, Standard scaling appears likely.

## Time series of scaled AR, Latcent, Area and kurtosis
Below the time series of the scaled aspect ratio (ar), centroid latitude (latcent), area, and kurtosis are displayed. First the data are reduced to DJFM and then scaled with the Standard Scaler.
The kurtosis is dominated by a few outliers and is therefore ommitted for the remainder of the analysis. Instead, the wind speed and edge are added. The wind speed serves as an estimator for the strength of a vortex.

In [ ]:
from sklearn.preprocessing import StandardScaler

# scale on entire data
sc = StandardScaler()
# time constraints
nwp_winter = nwp_all[nwp_all['month'].isin([12,1,2,3])]
nwp_winter.reset_index(drop=True, inplace=True)

# only geometric moments + wind speed
nwp_geo = nwp_winter[['string', 'year', 'month', 'day', 'area', 'ar', 'latcent', 'kurtosis', 'u', 'edge']]
# scale on DJFM
foi = ['scaled_area', 'scaled_ar', 'scaled_latcent', 'scaled_kurtosis', 'scaled_u', 'scaled_edge']
nwp_geo.loc[:, foi] = sc.fit_transform(nwp_geo[['area', 'ar', 'latcent', 'kurtosis', 'u', 'edge']])

In [ ]:
time_span = 2000
positions = [0]
positions[1: ] = [x for x in range(90, time_span, 121)]
for i in range(len(positions)):
    if nwp_geo.loc[positions[i], 'string'].year % 400 == 0:
        positions[i] = positions[i] +1
        continue
    if nwp_geo.loc[positions[i], 'string'].year % 4 == 0:
        positions[i] = positions[i] + 1
        continue

positions.append(time_span)

from vortexclust.workflows.demo import plot_timeseries_moments
plot_timeseries_moments(nwp_geo,
                        ['scaled_ar', 'scaled_latcent', 'scaled_area', 'scaled_kurtosis', 'scaled_u', 'scaled_edge'],
                        ['ar', 'latcent', 'area', 'kurtosis', 'u', 'edge'],
                        title = 'Vortex Geometric Moments (selected to DJFM, then scaled)',
                        time_span=time_span,
                        positions=positions,
                        num_plots=3,
                        figsize=(10,7.5),
                        savefig="../output/nwp/scaled_moments.png")

## Seasonality
Seasonality refers to recurring patterns or cycles in the data that occur at regular intervals. In climate data, such patterns are often driven by the solar cycle. For example, the stratospheric vortex builds due to a strong temperature gradient after the autumnal equinox and persists throughout the polar night. As polar day begins and the stratosphere warms, the polar vortex weakens and breaks down.

Neglecting seasonality can bias clustering results by grouping data points based on shared timeing rather than ib underlying physical characteristics. This is particularly problematic for clustering algorithms that are sensitive to the absolute distribution of the data and distances between samples. Removing or accounting for periodic signals ensures that resulting clusters represent true dynamical regimes of the vortex rather than artifacts of seasonality.

### Durbin-Watson Test
The Durbin-Watson test is used to detect the presence of first order autocorrelation in the residuals of a regression model. It assumes that:

- The residuals (errors) are normally distributed with a mean of 0
- The residuals are stationary over time

Commonly, it is interpretated as follows:
- **0 to 1.5**: strong **positive autocorrelation**
- **1.5 to 2.5**: little to **no autocorrelation**
- **2.5 to 4**: strong **negative autocorrelation**

In this analysis, the Durbin-Watson test indicates strong positive autocorrelation in the data, consistent with the findings of H11. This supports the presence of periodic signals in the data.

In [ ]:
from statsmodels.stats.stattools import durbin_watson

for feature in foi:
    dw = durbin_watson(nwp_geo[feature])
    print(f"Durbin-Watson statistic on {feature}: {np.round(dw, 3)}")

### Autocorrelation function
To further investigate seasonality, the autocorrelation function (ACF) is used. The ACF measures how well a time series correlates with lagged versions of itself. It is defined at lag $k$ as follows

$$\rho_k = \frac{\mathrm{Cov}(x_t, x_{t-k})}{\sigma(x_t) \sigma(x_{t-k})}$$

In this analysis, the ACF is computed on unit vector normalized data. Importantly, scaling does not affect the shape of the autocorrelation, as it is inherently scale-invariant.

The dashed horizontal lines in the plots represent the 5% significance bounds of the ACF. Values of autocorrelation that lie above or below these bounds indicate statistically significant correlation at the corresponding lag.

⚠ For large lag values, ACF computation can become slow. It is recommended to use lags between 500 and 1500.

#### Interpretation
All features show strong seasonal behaviour over the entire timeseries, which is to be expected from the formation and break down behaviour of the polar vortex.

The autocorrelation structure of the zonal wind speed (u) and scaled edge exhibit strong periodic behaviors, with clear peaks at roughly 120-days lags. This indicates strong periodic signals, which is consistent with the extremely low Durbin-Watson statistic (0.033 and 0.058, respectively).

In contrast, all other features show no pronounced periodicity in the autocorrelation function. Instead, they display low-amplitude short-range fluctuations and a relatively fast decay of correlation. While the Durbin-Watson statistic for them also indicates positive autocorrelation, it likely reflects smoother structural variability or low-frequency trends rather than a strictly seasonal signal.

These diagnostics support the application of deseasonalization techniques, such as SSA or EEOF, to wind speed and edge prior to clustering. For geometric variables like area, ar, latcent, no periodicity was found with the autocorrelation function. Hence, no deseasonalization is applied.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

fig, ax = plt.subplots(2,2, figsize=(12, 10))
ax = ax.flatten()

plot_acf(nwp_geo['scaled_edge'], lags=500, ax=ax[0],  marker=None)
plot_acf(nwp_all['edge'], lags=500, ax=ax[1], marker=None)

plot_acf(nwp_geo['scaled_u'], lags=500, ax=ax[2], marker=None)
plot_acf(nwp_all['u'], lags=500, ax=ax[3], marker = None)

ax[0].set_title('Autocorrelation of edge (DJFM)')
ax[1].set_title('Autocorrelation of edge (entire timeseries)')
ax[2].set_title("Autocorrelation of wind speed 'u' (DJFM)")
ax[3].set_title("Autocorrelation of wind speed 'u' (entire timeseries)")

for i in range(4):
    # 5% boundaries
    ax[i].axhline(y=0.05, linestyle='--', color='black', linewidth=1)
    ax[i].axhline(y=-0.05, linestyle='--', color='black', linewidth=1)
    # limit at lag
    ax[i].set_xlim(0, 500)
    # reset xticks to 120days period
    ax[i].set_xticks(positions[:5])
    ax[i].set_xticklabels([str(x) for x in positions[:5]])

    # set dates as second axis
    ax_top = ax[i].twiny()
    ax_top.set_xlim(ax[i].get_xlim())
    ax_top.set_xticks(positions[:5])
    if i%2 == 0:
        labels_dt = nwp_geo.iloc[positions[:5]]['string'].dt.strftime("%m-%y")
    else:
        labels_dt = nwp_geo.iloc[positions[:5]]['string'].dt.strftime("%m-%y")
    ax_top.set_xticklabels(labels_dt, ha='center', rotation=0)

plt.tight_layout()
plt.savefig("../output/nwp/acf.png")
plt.show()

### Singular Spectrum Analysis

Singular Spectrum Analysis (SSA) is a non-parametric spectral decomposition method that can be understood as applying singular value decomposition (SVD) to a lagged embedding of a time series. The original univariate time series is first transformed into a multivariate series via delay embedding, using a sliding window of length $M$.

The choice of window length $M$ is critical: larger values allow for finer decomposition, but for periodic signals, $M$ should not exceed the dominant period to avoid over-smoothing or leakage. Once the embedding is constructed, SVD is applied to extract principal components (eigenvectors) that represent the dominant patterns or oscillatory modes within the time series.

The original signal can then be reconstructed as a sum of selected components, allowing separation of trend, oscillatory, and noise-like behavior.

In [ ]:
from pyts.decomposition import SingularSpectrumAnalysis

M = 120
ssa = SingularSpectrumAnalysis(window_size=M)
# ssa_area = ssa.fit_transform(nwp_geo['scaled_area'].values.reshape(1,-1))
ssa_edge = ssa.fit_transform(nwp_geo['scaled_edge'].values.reshape(1, -1))
ssa_u = ssa.fit_transform(nwp_geo['scaled_u'].values.reshape(1, -1))

fig, axes = plt.subplots(2, figsize=(12, 5))
# axes[2].set_title('SSA and original data of area (DFJM)')
axes[0].set_title('SSA and original data of wind speed (DFJM)')
axes[1].set_title('SSA and original data of edge (DFJM)')
# axes[2].plot(nwp_geo['scaled_area'][:1000], label='original')
axes[0].plot(nwp_geo['scaled_u'][:1000], label='original')
axes[1].plot(nwp_geo['scaled_edge'][:1000], label='original')
for i in range(2):
    # axes[2].plot(ssa_area[i, :1000], label=f"SSA {i}")
    axes[0].plot(ssa_u[i, :1000], label=f"SSA {i}")
    axes[1].plot(ssa_edge[i, :1000], label=f"SSA {i}")
axes[0].legend(loc='upper right')
axes[1].legend(loc='upper right')
# axes[2].legend(loc='upper right')
for i in range(2):
    # set dates as second axis
    ax_top = axes[i].twiny()
    ax_top.set_xlim(axes[i].get_xlim())
    ax_top.set_xticks(positions[:10])
    axes[i].set_xticks(positions[:10])
    labels_dt = nwp_geo.iloc[positions[:10]]['string'].dt.strftime("%m-%y")
    ax_top.set_xticklabels(labels_dt, ha='center', rotation=0)

plt.tight_layout()
plt.show()

### Extended Empirical Orthogonal Function
The Extended Empirical Orthogonal Function (EEOF/EOF) is an extension of the standard EOF analysis, adapted for spatiotemporal data. Like SSA, EEOF decomposes time series into orthogonal components, but it does so across multiple correlated variables or spatial locations, incorporating lagged temporal information into each observation vector. This makes EEOF especially well suited for capturing propagating modes and phase relationsships between variables.

In the EEOF framework, the data matrix is constructed by stacking lagged versions of each variable (or spatial point), producing a higher-dimensional trajectory matrix. A singular value decomposition (SVD) is then performed to extract the dominant orthogonal modes. These modes can be interpreted as coherent structures in space-time and used for reconstruction, filtering, or dimensionality reduction prior to clustering or regression.

In the implementation below, EEOF decomposition is applied to the scaled DJFM time series of zonal wind speed and vortex area. A lag window of 400 days is selected, which captures multiple full seasonal cycles and is suitable for identifying low-frequency variability. The first 30 principal components are extracted. The explained variance ratio provides insight into the relative importance of each component, while the reconstructed signals can be used for seasonality filtering or further dynamical interpretation. Plotting the EEOF and the Empirical Principle Components (EPCs) over time as well as their phase diagram, gives insight into in the stability of the periodic signal.

In [ ]:
epc_u, eeof_u, expl_var_ratio_u, reconstructed_u, _ = vortexclust.compute_eeof(nwp_geo['scaled_u'], M=400, n_components=30)
epc_edge, eeof_edge, expl_var_ratio_edge, reconstructed_edge, _ = vortexclust.compute_eeof(nwp_geo['scaled_edge'], M=400, n_components=30)

from vortexclust.workflows.demo import plot_eeof
print('The EEOF evaluation for wind speed based on NWP data.')
plot_eeof(epc_u, eeof_u, expl_var_ratio_u, savefig="../output/nwp/eeof_u.png")
print('The EEOF evaluation for edge based on NWP data.')
plot_eeof(epc_edge, eeof_edge, expl_var_ratio_edge, savefig="../output/nwp/eeof_edge.png")

### Filtering
The EEOF analysis reveals different structures in the two variables. For the edge variable, the first two components resemble sinusoidal modes offset by phase — a strong indicator of underlying periodicity. These patterns closely match those reported in H11 for the area signal, confirming the presence of a seasonal cycle.

For wind speed, the EEOF structure differs slightly from the ERA5 analysis. While ERA5 data showed two dominant modes followed by two weaker ones, the NWP eigenvalue spectrum exhibits four leading components, together explaining ~40% of the variance. This indicates a broader spread of dominant modes.

To isolate the seasonal component, we follow H11 and reconstruct the signal using the leading four components of both EEOF and SSA. For comparison, reconstructions using 2, 4, and 30 components are shown. Including too many components leads to near-complete reconstruction of the original series, leaving little meaningful signal after filtering. Therefore, we proceed with the first four components for both edge and wind speed.

In [ ]:
fig, ax = plt.subplots(4, figsize=(12, 10), sharex='all')

ax[0].set_title('Reconstruction of edge from SSA')
ax[1].set_title("Reconstruction of wind speed from SSA")
ax[2].set_title('Reconstruction of edge from EEOF')
ax[3].set_title("Reconstruction of wind speed from EEOF")

ax[0].plot(nwp_geo['scaled_edge'][:1000], label='original')
ax[1].plot(nwp_geo['scaled_u'][:1000], label='original')
ax[2].plot(np.arange(1, 1001), nwp_geo['scaled_edge'][:1000], label='original')
ax[3].plot(np.arange(1, 1001), nwp_geo['scaled_u'][:1000], label='original')
for i in [4, 6, 30]:
    ssa_reconstructed_edge = ssa_edge[:i].sum(axis = 0)
    ssa_reconstructed_u = ssa_u[:i].sum(axis=0)
    _, _, _, eeof_reconstructed_edge, _ = vortexclust.compute_eeof(nwp_geo['scaled_edge'], M=400, n_components=i)
    _, _, _, eeof_reconstructed_u, _ = vortexclust.compute_eeof(nwp_geo['scaled_u'], M=400, n_components=i)
    ax[0].plot(ssa_reconstructed_edge[:1000], label=f"{i}")
    ax[1].plot(ssa_reconstructed_u[:1000], label=f"{i}")
    ax[2].plot(eeof_reconstructed_edge[399:1399, 0], label=f"{i}")
    ax[3].plot(eeof_reconstructed_u[399:1399, 0], label=f"{i}")

for i in range(4):
    ax[i].legend(loc='upper right')
    ax[i].axvline(x=484, color='black', linestyle='--')
# set dates as second axis
ax_top = ax[0].twiny()
ax_top.set_xlim(ax[0].get_xlim())
ax_top.set_xticks(positions[:10])
ax[0].set_xticks(positions[:10])
labels_dt = nwp_geo.iloc[positions[:10]]['string'].dt.strftime("%m-%y")
ax_top.set_xticklabels(labels_dt, ha='center', rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# compute eeof with respective number of components
_, _, _, eeof_reconstructed_edge, _ = vortexclust.compute_eeof(nwp_geo['scaled_edge'], M=400, n_components=4)
_, _, _, eeof_reconstructed_u, _ = vortexclust.compute_eeof(nwp_geo['scaled_u'], M=400, n_components=4)

# shorten nwp data by 400 days
nwp_geo = nwp_geo.iloc[:-399, :]

nwp_geo.loc[:, 'ssa_filtered_edge'] = nwp_geo['scaled_edge'] - ssa_edge[:4, :nwp_geo.shape[0]].sum(axis = 0).T
nwp_geo.loc[:, 'ssa_filtered_u'] = nwp_geo['scaled_u'] - ssa_u[:4, :nwp_geo.shape[0]].sum(axis=0).T
nwp_geo.loc[:, 'eeof_filtered_edge'] = nwp_geo['scaled_edge'] - eeof_reconstructed_edge[399:, 0].T
nwp_geo.loc[:, 'eeof_filtered_u'] = nwp_geo['scaled_u'] - eeof_reconstructed_u[399:, 0].T

fig, ax = plt.subplots(2, figsize=(12, 5), sharex='all')
ax[0].set_title('Edge and filtered edge')
ax[0].plot(nwp_geo['scaled_area'][:1000], label='original')
ax[0].plot(nwp_geo['ssa_filtered_edge'][:1000], '-.', label='Filtered with SSA')
ax[0].plot(nwp_geo['eeof_filtered_edge'][:1000], '--', label='Filtered with EEOF')
ax[1].set_title('Wind speed and filtered wind speed')
ax[1].plot(nwp_geo['scaled_u'][:1000], label='original')
ax[1].plot(nwp_geo['ssa_filtered_u'][:1000], '-.', label='Filtered with SSA')
ax[1].plot(nwp_geo['eeof_filtered_u'][:1000], '--', label='Filtered with EEOF')

for i in range(2):
    ax[i].legend(loc='upper right')
    ax[i].axvline(x=484, color='black', linestyle='--')
# set dates as second axis
ax_top = ax[0].twiny()
ax_top.set_xlim(ax[0].get_xlim())
ax_top.set_xticks(positions[:10])
ax[0].set_xticks(positions[:10])
labels_dt = nwp_geo.iloc[positions[:10]]['string'].dt.strftime("%m-%y")
ax_top.set_xticklabels(labels_dt, ha='center', rotation=0)
plt.tight_layout()
plt.savefig("../output/nwp/ssa_eeof_filter.png")
plt.show()

## Gap statistic
The performance of clustering algorithm depends severely on the right number of clusters as input. Determining the optimal number of cluster $k_{opt}$ is nontrivial, and fitting models is computationally expensive. Hence, more elegant ways than simply testing different $k$ are required. Further unsupervised clustering has no notion of wrong or right, but requires different measures to evaluate a models quality. Most techniques define a measure of similarity, which should be maximized in clusters and minimized between clusters. Consequently, dissimilarity is defined vice versa. Then the similarity or dissimilarity is compared for different values of $k$.

The gap statistic introduced by Tibshirani et al. (2001) bases on the idea to compare the within cluster dispersion of the actual data to a null reference distribution. The null reference is typically generated by a homogeneous Poisson point process (HPPP).

To compute the within cluster dispersion, the pairwise squared distances between all data points in a cluster $C_m, m=1, ..., k$ are computed by

$$D_m = \sum_{i,j \in C_m} \|x_i- x_j\|^2 = 2n_m \sum_{i=1}^{n_m} \|x_i- \mu_m\|^2$$

with $|C_m| = n_m$ being the size of the $m$th cluster. Note that this assumes Euclidean geometry and unweighted samples.

Some implementations use the distance $\|x_i - c_m\|$, i.e. the distance of a data point $x_i$ to its corresponding cluster center $c_m=\mu_m$. This is only applicable in clustering algorithms that have some notion of centers such as k-means. For clustering methods without such centers, such as hierarchical clustering, the gap statistic is more precise using the above notation instead of calculating centers manually.

In the next step, $D_m$ is averaged to obtain the overall dispersion index $W_k$.
$$W_k = \sum_{m=1}^{k} \frac{D_m}{2n_m}$$
The sum of squared distances to the center of a cluster is the same as $W_k$ as long there are no sample weights.

Finally, the gap statistic for each $k$ is computed by the difference between the *within dispersion index* of the data, $log(W_k)$,  and that expected from the generated reference $log(W_k^*)$.
$$G(k) = E[log(W_k^*)] - log(W_k)$$

Here, $E(\cdot)$ is the expectation as usually defined. A Monte Carlo simulation generates $N$ samples of $log(W_k^*)$, where each $W_k^*$ is based on the same number of points as the original data. Additionally, the standard deviation $s_k^*$ is computed and used to account for the simulation error in $E[log(W_k^*)]$ with the inflated standard deviation $s_k = s_k^*\sqrt{1+\frac{1}{N}}$. Finally, the optimal $k_{opt}$ is determined by the smallest $k$ such that

$$
	G(k) \geq G(k+1) - s_{k+1}
$$

Tibshirani et al. demonstrated that the gap statistic is a robust method for estimating the number of clusters. Their findings indicate that, for elongated clusters, the gap statistic yields better results with a null reference aligned to the data's principal components. In contrast, using a simple uniform distribution leads to inferior results. Overall, the underlying distribution of data does not influence the results of the gap statistic, making it more robust than other methods.

The performance of the gap statistic deteriorates in high-dimensional settings due to computational costs. It is also prone to underestimation the optimal $k$ in settings with many clusters. The underestimation vanishes with increasing dimensionality.

H11 focused primarily on the gap statistic. In the analysis below, it is applied alongside the **elbow** and **silhouette** methods for comparison.

## Hierarchical Clustering
Hierarchical clustering is a family of clustering methods that focuses on building nested clusters. This is achieved by merging or splitting them successively. The hierarchy can be represented in a so-called dendrogram, i.e. a tree, where the root contains all samples and each leaf represents a samples. Leaves next to each other correspond to similar samples. H11 does not specify the precise algorithm, so agglomorative clustering is applied here as an adaption.

Agglomerative clustering uses a bottom up approach, i.e. each sample starts as its own cluster and is successively merged with the nearest cluster. The mergingn criterion is defined by a linkage method, which determines how distances between clusters are computed. The default is $complete$ as determined in H11 to prevent spheroidal or elongated (chained) clusters and distance means the Euclidean distance. Below an overview of some linkage methods is given.

|               | between points in Cluster | to different clusters                      |
|---------------|---------------------------|--------------------------------------------|
| Closest       | Single Linkage            | Centroid Linkage                           |
| Most Distance | **Complete Linkage**      | Average Linkage (Average of all distances) |

Due to the computational intensity of model fitting and validation, the following code cells may require several minutes to complete.

In [ ]:
k_max = 10
gap_ar_latcent = vortexclust.gap_statistic(nwp_geo[['scaled_ar', 'scaled_latcent']], k_max=k_max, n_replicates=15)
gap_ar_latcent_area = vortexclust.gap_statistic(nwp_geo[['scaled_ar', 'scaled_latcent', 'scaled_area']], k_max=k_max, n_replicates=15)
gap_ar_latcent_area_edge = vortexclust.gap_statistic(nwp_geo[['scaled_ar', 'scaled_latcent', 'scaled_area', 'ssa_filtered_edge']], k_max=k_max, n_replicates=15)
gap_ar_latcent_u = vortexclust.gap_statistic(nwp_geo[['scaled_ar', 'scaled_latcent', 'ssa_filtered_u']], k_max=k_max, n_replicates=15)

In [ ]:
elbow_ar_latcent = vortexclust.elbow_method(nwp_geo[['scaled_ar', 'scaled_latcent']], k_max=k_max)
elbow_ar_latcent_area = vortexclust.elbow_method(nwp_geo[['scaled_ar', 'scaled_latcent', 'scaled_area']], k_max=k_max)
elbow_ar_latcent_area_edge = vortexclust.elbow_method(nwp_geo[['scaled_ar', 'scaled_latcent', 'scaled_area', 'ssa_filtered_edge']], k_max=k_max)
elbow_ar_latcent_u = vortexclust.elbow_method(nwp_geo[['scaled_ar', 'scaled_latcent', 'ssa_filtered_u']], k_max=k_max)

In [ ]:
silhouette_ar_latcent = vortexclust.silhouette_method(nwp_geo[['scaled_ar', 'scaled_latcent']], k_max=k_max)
silhouette_ar_latcent_area = vortexclust.silhouette_method(nwp_geo[['scaled_ar', 'scaled_latcent', 'scaled_area']], k_max=k_max)
silhouette_ar_latcent_area_edge = vortexclust.silhouette_method(nwp_geo[['scaled_ar', 'scaled_latcent', 'scaled_area', 'ssa_filtered_edge']], k_max=k_max)
silhouette_ar_latcent_u = vortexclust.silhouette_method(nwp_geo[['scaled_ar', 'scaled_latcent', 'ssa_filtered_u']], k_max=k_max)

In [ ]:
fig, ax = plt.subplots(2,2, figsize=(15, 10), sharex='all')
ax[0][0].errorbar(np.arange(1,k_max+1), gap_ar_latcent[:, 0], yerr=gap_ar_latcent[:, 1], label='AR, Latcent')
ax[0][0].errorbar(np.arange(1,k_max+1), gap_ar_latcent_area[:, 0], yerr=gap_ar_latcent_area[:, 1], label='AR, Latcent, Area')
ax[0][0].errorbar(np.arange(1,k_max+1), gap_ar_latcent_u[:, 0], yerr=gap_ar_latcent_u[:, 1], label='Ar, Latcent, Filtered wind speed')
ax[0][0].errorbar(np.arange(1,k_max+1), gap_ar_latcent_area_edge[:, 0], yerr=gap_ar_latcent_area_edge[:, 1], label='Ar, Latcent, Area, Edge')
ax[0][0].set_title('Gap statistic')
ax[0][0].legend(loc='upper right')

ax[1][0].plot(np.arange(1,k_max+1), elbow_ar_latcent[0], label='AR, Latcent')
ax[1][0].plot(np.arange(1,k_max+1), elbow_ar_latcent_area[0], label='AR, Latcent, Area')
ax[1][0].plot(np.arange(1,k_max+1), elbow_ar_latcent_u[0], label='AR, Latcent, Filtered wind speed')
ax[1][0].plot(np.arange(1,k_max+1), elbow_ar_latcent_area_edge[0], label='AR, Latcent, Area, Edge')
ax[1][0].set_title('Elbow method - Distortion')
ax[1][0].legend(loc='upper right')

ax[1][1].plot(np.arange(1,k_max+1), elbow_ar_latcent[1], label='AR, Latcent')
ax[1][1].plot(np.arange(1,k_max+1), elbow_ar_latcent_area[1], label='AR, Latcent, Area')
ax[1][1].plot(np.arange(1,k_max+1), elbow_ar_latcent_u[1], label='AR, Latcent, Filtered wind speed')
ax[1][1].plot(np.arange(1,k_max+1), elbow_ar_latcent_area_edge[1], label='AR, Latcent, Area, Edge')
ax[1][1].set_title('Elbow method - Inertia')
ax[1][1].legend(loc='upper right')

ax[0][1].plot(np.arange(2,k_max+1), silhouette_ar_latcent, label='AR, Latcent')
ax[0][1].plot(np.arange(2,k_max+1), silhouette_ar_latcent_area, label='AR, Latcent, Area')
ax[0][1].plot(np.arange(2,k_max+1), silhouette_ar_latcent_u, label='AR, Latcent, Filtered wind speed')
ax[0][1].plot(np.arange(2,k_max+1), silhouette_ar_latcent_area_edge, label='AR, Latcent, Area, Edge')
ax[0][1].set_title('Silhouette method')
ax[0][1].legend(loc='upper right')

plt.suptitle('Different k determination methods (NWP)')
plt.xticks(np.arange(1, k_max+1))
fig.text(0.5, 0, 'Number of clusters', ha='center')
plt.tight_layout()
plt.savefig("../output/nwp/Kopt_nwp_complete.png")
plt.show()

In [ ]:
print("Optimal number of clusters by each method: ")
p_1, p_2, p_3, p_4 = 1,1,1,1
for k in range(1, 10):
    if p_1 and (gap_ar_latcent[k][0] >= gap_ar_latcent[k+1][0] - gap_ar_latcent[k+1][1]):
        print('Gap statistic (AR, Latcent): ', k+1) # index starts at 0, k starts at 1
        p_1=0
    if p_2 and (gap_ar_latcent_area[k][0] >= gap_ar_latcent_area[k+1][0] - gap_ar_latcent_area[k+1][1]):
        print("Gap statistic (AR, Latcent, Area): ", k+1)
        p_2 = 0
    if p_4 and (gap_ar_latcent_area_edge[k][0] >= gap_ar_latcent_area_edge[k+1][0] - gap_ar_latcent_area_edge[k+1][1]):
        print("Gap statistic (AR, Latcent, Area, Edge): ", k+1)
        p_4 = 0
    if p_3 and (gap_ar_latcent_u[k][0] >= gap_ar_latcent_u[k+1][0] - gap_ar_latcent_u[k+1][1]):
        print("Gap statistic (AR, Latcent, filtered wind speed): ", k+1)
        p_3=0

print("Silhouette method (AR, Latcent): ", pd.DataFrame(silhouette_ar_latcent).idxmax()[0]+2)
print("Silhouette method (AR, latcent, Area): ", pd.DataFrame(silhouette_ar_latcent_area).idxmax()[0]+2)
print("Silhouette method (AR, latcent, Area, Edge): ", pd.DataFrame(silhouette_ar_latcent_area_edge).idxmax()[0]+2)
print("Silhouette method (AR, latcent, filtered wind speed): ", pd.DataFrame(silhouette_ar_latcent_u).idxmax()[0]+2)

### Optimal number of clusters
The gap statistic suggest that the data lacks inherent clustering structure, consistently favouring $k=1$. The silhouette method partially supports that.

However, when assuming that at least some structure exists and evaluating $k > 1$, all methods indicate $k_{opt} = 3$ when clustering based on ar and latcent. The model based on ar, latcent and the filtered wind speed clearly indicates $k_{opt} = 4$, exception is the silhouette method. The silhouette methods favours clearly separated clusters of the same density and size. In the analysis of ERA5 reanalysis data, it showed that a very sparse large cluster emerged from the data. Should this occur in the NWP data as well, the silhouette method might be underestimating the optimal number of clusters.

In contrast, the introduction of the area computed from $Z10$ leads to ambiguity. The gap statistic and silhouette method indicate no structure in the data. Though the gap statistic has local maxima at $k=6$ with ar, latcent, and area, and $k=7$, when the edge is added. It is a known disadvantage of the gap statistic, that it will select the first local maxima as optimal $k$, even though another $k$ might be more suitable. The elbow methods support these results.

Tabular summary of $k_{opt}$ for each method:

| Used features           | Gap statistic | Silhouette  | Elbow (Distortion) | Elbow (Inertia) | $k_{opt}$ |
|-------------------------|---------------|-------------|--------------------|-----------------|-----------|
| AR, latcent             | 1 or 3        | 2 or 3      | 3 or 7             | 3 or 7          | 3         |
| AR, latcent, wind speed | 1 or 4 or 6   | 2           | 4                  | 4               | 4         |
| AR, latcent, area       | 3 or 6        | 2 or 3 or 6 | 6                  | 6               | 6         |
| AR, latcent, area, edge | 1 or 6 or7    | 2 or 3 or 4 | 6 or 7             | 6 or 7          | 7         |

For the remainder of the analysis $k_{opt}$ as indicated above is adopted, based on convergence of the elbow method, gap statistic and silhouette method.

In [ ]:
from sklearn import clone
from sklearn.cluster import AgglomerativeClustering
import vortexclust.visualization as viz

features_kopt = [{'features' : ['scaled_ar', 'scaled_latcent'], 'k_opt' : 3, 'line':6},
                 {'features' : ['scaled_ar', 'scaled_latcent', 'scaled_area'], 'k_opt' : 6, 'line':6.8},
                 {'features' : ['scaled_ar', 'scaled_latcent', 'scaled_area', 'ssa_filtered_edge'], 'k_opt' : 7, 'line':7},
                 {'features' : ['scaled_ar', 'scaled_latcent', 'ssa_filtered_u'], 'k_opt' : 4, 'line':7}]

Y = []
base_model = AgglomerativeClustering(linkage='complete', compute_distances=True)

fig = plt.figure(figsize=(15,10))
for idx, feat_k in enumerate(features_kopt):
    model = clone(base_model)
    model.set_params(n_clusters = feat_k['k_opt'])
    model.fit(nwp_geo[feat_k['features']])
    ax = fig.add_subplot(2,2,idx+1)
    ax.set_title(str(feat_k['features'])[1:-1]+", k="+str(feat_k['k_opt']))
    viz.plot_dendrogram(model, truncate_mode='level', p=4, direction='LR')
    ax.axvline(feat_k['line'], ls='--', color='black')
    y_pred = model.labels_.astype(int)
    Y.append(y_pred)
plt.tight_layout()
plt.savefig("../output/nwp/dendrogram_complete.png")
plt.show()

fig = plt.figure(figsize=(15,10))
for idx, feat_k in enumerate(features_kopt):
    if len(feat_k['features']) == 3:
        ax = fig.add_subplot(2,2,idx+1, projection='3d')
        ax.scatter(nwp_geo[feat_k['features']].iloc[:, 0],
                   nwp_geo[feat_k['features']].iloc[:, 1],
                   nwp_geo[feat_k['features']].iloc[:, 2],
                   c=Y[idx], cmap='tab10')
        ax.set_facecolor((0, 0, 0, 0))
        ax.set_xlabel(feat_k['features'][0])
        ax.set_ylabel(feat_k['features'][1])
        ax.set_zlabel(feat_k['features'][2])
    else:
        ax = fig.add_subplot(2,2, idx+1)
        ax.scatter(nwp_geo[feat_k['features']].iloc[:, 0],
                   nwp_geo[feat_k['features']].iloc[:, 1],
                   c=Y[idx], cmap='tab10')
        ax.set_xlabel(feat_k['features'][0])
        ax.set_ylabel(feat_k['features'][1])
    plt.subplots_adjust(hspace=0.4, wspace=0.4)
    ax.set_title(str(feat_k['features'])[1:-1]+", k="+str(feat_k['k_opt']))
plt.show()

# Results
Below the relative and absolute distribution of physical measurements are visualized for each cluster. The goal is to identify and assign clear characteristics for each cluster. In particular, the following is expected:
- A split cluster (S) that corresponds to the threshold based assignment of split,
- A displaced cluster (D) characterized by notably low centroid latitude,
- A normal or undisturbed cluster (U) without strong anomalies.

Note that split events are usually accompanied by displacement. Models with more than 3 clusters are expected to have variations of vortex regimes with large area, or further distinctions between split and displaced events.

All models with more than 3 clusters created a cluster with a single sample. This should be avoided, but reducing the number of clusters did not change the results. Using other linkages could be a solution, as this behaviour is a known fault of single, average and complete linkages. In particular, single linkage encourages the creation of large clusters and macroscopic small ones. Using `ward` linkage solve this and is adjusted in an additional section.

## Statistical Profiles of Clusters
### Aspect ratio and Latcent
The majority of samples falls into Class **2**, which does not show any remarkable characteristics. Class **1** exhibits a significantly larger ar than the other two clusters, which does have a minimum of $ar_{scaled}=2.655$ (resp. $ar = 2.257$). Class **0** displays a significantly smaller latcent, barely exceeding the average latcent at 0. Consequently, classes are assigned as follows:

**0** &rarr; S (Split)<br>
**1** &rarr; U (Undisturbed/normal state)<br>
**2** &rarr; D (Displaced)

This is the only model that did not create clusters with only one sample.

In [ ]:
from vortexclust.workflows.demo import plot_hist_per_class

y_names = ['y_ar_latcent', 'y_ar_latcent_scArea', 'y_ar_latcent_area_edge', 'y_ar_latcent_u']
nwp_geo[y_names] = pd.DataFrame(Y).T

nwp_geo['y_ar_latcent'] = nwp_geo['y_ar_latcent'].replace({0: 'S', 1:'D', 2:'U'})
print("Averages per class and features:")
print(nwp_geo[['y_ar_latcent', 'scaled_ar', 'scaled_latcent', 'ar']].groupby(['y_ar_latcent']).mean())
print(nwp_geo['y_ar_latcent'].value_counts())
plot_hist_per_class(nwp_geo, # data
                    features_kopt[0], # information about used feature and k_opt
                    'y_ar_latcent',
                    savefig="../output/nwp/hist_nwp_mBasic_") # column name with y values

### Aspect ratio, latcent and wind speed
Including the wind speed enhanced the separation between classes in the full analysis replicating H11, but results in less pronounced results in this case. The majority of samples falls into Class **1**, which is characterised by relatively average values for latcent and ar. Interestingly, it has higher values for the filtered wind speed than the other two meaningful classes. Class **2** exhibits a significantly larger ar and class **0** displays a lower centroid latitude. Assigning the S, D, and U respectively, it matches the observation that **2** and **0** have similar distributions for the wind speed, but generally weaker winds than **1**. Split and disturbed events usually result in weakened westerlies or even their reversal. Consequently, classes are assigned as follows:

**0** &rarr; D (Displaced)<br>
**1** &rarr; U (Undistributed/normal state)<br>
**2** &rarr; S (Split)<br>
**3** &rarr; E_ar (Extrema with large aspect ratio)

The fourth class which was detected when determining the optimal k consists of one sample with an extremely large aspect ratio ($ar=4.37, ar_{scaled} = 9.35$)and is therefore marked as **E** for extrema. It should be classified into the split cluster, but due to scaling its distance to the other clusters is enlarged.

In [ ]:
nwp_geo['y_ar_latcent_u'] = nwp_geo['y_ar_latcent_u'].replace({0:'D', 1:'U', 2:'S', 3:'E'})
print("Averages per class and features:")
print(nwp_geo[['y_ar_latcent_u', 'scaled_ar', 'scaled_latcent', 'ssa_filtered_u', 'ar']].groupby(['y_ar_latcent_u']).mean())
print(nwp_geo[['y_ar_latcent_u', 'scaled_ar', 'scaled_latcent', 'ssa_filtered_u']].groupby(['y_ar_latcent_u']).size())

# exclude extrema from plot
plot_hist_per_class(nwp_geo.drop(nwp_geo[nwp_geo['scaled_ar'] > 9].index), # data
                    features_kopt[3], # information about used feature and k_opt
                    'y_ar_latcent_u',
                    savefig="../output/nwp/hist_nwp_mWind_") # column name with y values

### Aspect ratio, latcent, and vortex area
Adding the area computed with at 10hPa geopotential height, added some ambiguity to the interpretation as split, displaced and undisturbed. Additionally, the optimal k was determined as 6. As in the previous model, a single example was extracted as its own cluster. Here, class **3** is a sample with an extremely large area, and class **4** a sample with an extremely large aspect ratio.
The remaining clusters could be characterized as split (class **2**) by the distribution of the aspect ratio, large (class **0**) by characterized by large vortex areas, displaced (class **1**) by small latcent values and undisturbed (class **5**) with relatively average values in all features. Hence, each class is marked as below.

Using ar, latcent, and area:<br>
**0** &rarr; L (Large area and slightly displaced)<br>
**1** &rarr; D (Displaced)<br>
**2** &rarr; S (Split)<br>
**3** &rarr; E_area (Extrema with large area, single sample)<br>
**4** &rarr; E_ar (Extrema with large aspect ratio, single sample)<br>
**5** &rarr; U (Undisturbed/normal state)<br>

In [ ]:
nwp_geo['y_ar_latcent_scArea'] = nwp_geo['y_ar_latcent_scArea'].replace({0: 'L', 1:'D', 2:'S', 3 : 'E_area', 4 : 'E_ar', 5 : 'U'})
print("Averages per class and features:")
print(nwp_geo[['y_ar_latcent_scArea', 'scaled_ar', 'scaled_latcent', 'scaled_area']].groupby(['y_ar_latcent_scArea']).mean())
print(nwp_geo[['y_ar_latcent_scArea', 'scaled_ar', 'scaled_latcent', 'scaled_area']].groupby(['y_ar_latcent_scArea']).size())

plot_hist_per_class(nwp_geo.drop(nwp_geo[(nwp_geo['scaled_ar'] > 9) | (nwp_geo['scaled_area'] > 14.5) ].index), # data
                    features_kopt[1], # information about used feature and k_opt
                    'y_ar_latcent_scArea',
                    savefig="../output/nwp/hist_nwp_mscArea_") # column name with y values

### Aspect ratio, latcent, area and edge

When the edge was added as additional information, even more clusters emerge and the optimal k was found to be 7.  Again, extrema where assigned their own classes, class **3** is a sample with an extremely large area, and class **5** contains a sample with an extremely large aspect ratio.
The remaining clusters are characterized as split (class **2**) by the distribution of the aspect ratio. Class **0** is described by vortices with small latcent values and significantly larger areas. This cluster seems to correspond closely to the identified **L** cluster in the full analysis in the other notebook. Class **1** contains roughly a third of the data, and seems to correspond to the displaced events as indicated by the latcent distribution. Further displaced vortices are contained in class **6**, though class **6** has an aspect ratio, that is above average. Hence, it is assigned **DS** for displaced and split state. Finally, class **4** represents the undisturbed state.

Using ar, latcent, area, and edge:<br>
**0** &rarr; L (Large)<br>
**1** &rarr; DI (Displaced I)<br>
**2** &rarr; S (Split)<br>
**3** &rarr; E_area (Extrema with large area, single sample)<br>
**4** &rarr; U (Undisturbed/normal state)<br>
**5** &rarr; E_ar (Extrema with large aspect ratio, single sample)<br>
**6** &rarr; DS (Displaced and split)<br>

In [ ]:
nwp_geo['y_ar_latcent_area_edge'] = nwp_geo['y_ar_latcent_area_edge'].replace({0: 'L', 1:'D', 2:'S', 3 : 'E_area', 4 : 'U', 5 : 'E_ar', 6 : 'DS'})

print("Averages per class and features:")
print(nwp_geo[['y_ar_latcent_area_edge', 'scaled_ar', 'scaled_latcent', 'scaled_area', 'ssa_filtered_edge']].groupby(['y_ar_latcent_area_edge']).mean())
print(nwp_geo[['y_ar_latcent_area_edge', 'scaled_ar', 'scaled_latcent', 'scaled_area', 'ssa_filtered_edge']].groupby(['y_ar_latcent_area_edge']).size())

plot_hist_per_class(nwp_geo.drop(nwp_geo[(nwp_geo['scaled_ar'] > 9) | (nwp_geo['scaled_area'] > 14.5)].index), # data
                    features_kopt[2], # information about used feature and k_opt
                    'y_ar_latcent_area_edge',
                    savefig="../output/nwp/hist_nwp_medge_") # column name with y values

## Physical interpretation and Literature Comparison

The following table illustrates the distribution of classes in H11:

<table>
  <tr>
    <th></th>
    <th colspan="4">AR, Latcent and filtered Area</th>
  </tr>
  <tr>
    <th>AR, Latcent</th>
    <th>D (0) </th>
    <th>U (2) </th>
    <th>S (1) </th>
    <th>Total</th>
  </tr>
  <tr>
    <th>D (2) </th>
    <td>7</td>
    <td>3</td>
    <td>0</td>
    <td>10</td>
  </tr>
  <tr>
    <th>U (1)</th>
    <td>4</td>
    <td>80</td>
    <td>2</td>
    <td>86</td>
  </tr>
  <tr>
    <th>S (0)</th>
    <td>1</td>
    <td>0</td>
    <td>3</td>
    <td>4</td>
  </tr>
  <tr>
    <th>Total</th>
    <td>12</td>
    <td>83</td>
    <td>5</td>
    <td>100</td>
  </tr>
</table>

When using wind speed as a proxy for vortex area, the clustering aligns closely with the class proportions reported by H11.

In contrast, the model using only AR and Latcent tends to underestimate split events, while slightly overrepresenting the displaced cluster. The inclusion of filtered wind speed corrects this over- and underestimation, resulting in a distribution that more faithfully reproduces the structure found in H11.

Notably, both approaches overestimate displaced events (D) by 5–14% compared to H11. This suggests that the centroid latitude alone may not be sufficient to isolate displacement events robustly, especially in cases when they occur with splits or large area structures.

For a better comparison, the cluster with the extrema is assigned to split.

In [ ]:
nwp_geo['y_ar_latcent_u'] = nwp_geo['y_ar_latcent_u'].replace({'E' : 'S'})

pd.pivot_table(data=nwp_geo[['y_ar_latcent', 'string','y_ar_latcent_u']], index='y_ar_latcent', columns='y_ar_latcent_u', aggfunc = 'count', margins=True)/nwp_geo.shape[0]*100

A reference classification of split versus displaced events was derived directly from the aspect ratio within the elliptical diagnostics. In this scheme, every event is categorized as either split or displaced, based solely on geometric thresholds.

All models are evaluated against this geometric classification to assess how well they capture the split regime. To quantify this, standard classification metrics are used: accuracy, precision, recall, and F1 score. These are defined based on:

To compare the classification by the hierarchical models to the computed split and displaced classification, the measures of accuracy (acc), precision, recall and f1 score are introduced. They base on the notion of
- true positive (TP), i.e. an even is clustered and computed as split,
- true negative (TN), i.e. an event is not clustered as split and not computed as such,
- false positive (FP), i.e. an event is clustered as split, but not computed as split,
- false negative (FN), i.e. an event is marked as not split, but was computed as split.

The four metrics are calculated as follows:
$$
acc = \frac{TN+TP}{TP+TN+FP+FN}
$$
$$
precision = \frac{TP}{TP+FP}
$$
$$
recall = \frac{TP}{TP+FN}
$$
$$
f1 = \frac{2*precision*recall}{precision+recall}
$$

All scores range from 0 (worst) to 1 (best). The minimal model based on the aspect ratio and centroid latitude achieves the best overall balance, with:
- accuracy = 0.99
- precision = 0.69
- recall = 0.94

All other models overestimates split events, resulting in a high number of false positives and a much lower precision. This highlights that a more specific distinction might be necessary when considering more features of the polar vortex.

In [ ]:
from vortexclust.workflows.demo import compare_cluster

compare_cluster(nwp_geo.merge(nwp_all[['string', 'form']], on='string', how='inner'), compare_col='form', pred_value='S', gt_value=1, y_names=y_names)

Seviour et al. (2013) presented a threshold based method to distinguish between split and displaced events. A displaced event is defined as:

> Displaced events are identified if the centroid latitude remains equatorward 66ıN for 7 days or more.7

And a split event as:
> Split events are identified if the aspect ratio remains above 2.4 for 7 days or more.

To avoid overlapping detections, no two events may occur within 30 days. If they do, the first occurrence determines the classification.

Compared to the results from the full analysis on the ERA5 data, the simulation seems to underrepresent split events. Only 0.24% of the events were classified as split based on the threshold above, while on the ERA5 reanalysis data 2.6% were identified as split.

Key results:

| Feature Set                          | Accuracy | Precision | Recall | F1 Score |
|--------------------------------------|----------|-----------|--------|----------|
| **AR, Latcent**                      | 0.99     | 0.13      | 0.94   | 0.23     |
| **AR, Latcent, Area**                | 0.97     | 0.09      | 1.00   | 0.16     |
| **AR, Latcent, Filtered Wind Speed** | 0.96     | 0.05      | 1.00   | 0.1      |
| **AR, Latcent, Area and Edge**       | 0.96     | 0.06      | 1.0    | 0.1      |

In comparison with this reference scheme, again, the model based on ar and latcent shows the best alignment.
All other models achieve perfect recall, but suffer from overclassification (precision < 0.1), mirroring its behaviour against the geometric `form` flag.

In [ ]:
labels = vortexclust.split_displaced_seviour(nwp_geo[['ar', 'latcent', 'string']], mark='all')
nwp_geo['seviour'] = labels

compare_cluster(nwp_geo, compare_col = 'seviour', pred_value = 'S', gt_value='split', y_names=y_names)

### Comparison to Major Warmings

Although not directly diagnostic of vortex morphology, major stratospheric warmings (MWs) are crucial for understanding sudden disruptions in vortex structure. The table below shows the distribution of cluster assignments across MW and non-MW periods, expressed as percentages of the full dataset.

Key Observations:
- Major Warmings (MW = 1) occur in ~2.48% of the dataset
- In all models, MWs are disproportionately associated with split, displaced, and large clusters.
    - For the AR + Latcent model, over 95% of MW events fall into the S or D clusters.
    - Respectively, the undisturbed (U) cluster dominates the non-MW periods.
- The filtered wind speed model maintains this pattern while reducing noise — MWs account for:
    - 2.23% of class D
    - 0.17% of class S
    - 0.08% of class U
- The 2 models accounting for the vortex area clearly associate MWs with large, displaced and split clusters.
    - As for the other 2 models, the majority of MW events coincides with displaced events.
    - The undisturbed and extreme states are not associated with MW.

These results are consistent with the physical understanding that MWs often coincide with major vortex disruptions, including splits and displacements. While not all splits are major warmings, and not all warmings result in splits, the alignment across models indicates that the clustering meaningfully captures dynamical states with increased MW likelihood.

In [ ]:
for y in y_names:
    print(np.round(pd.pivot_table(nwp_geo.merge(nwp_all[['string', 'MW']], on='string', how='inner')[['MW', y, 'string']], index='MW', columns=y, aggfunc='count', margins=True)/nwp_geo.shape[0]*100,2))

### Overview of descriptive statistics for each class

Below the statistics for all features are displayed. The minimum (resp. maximum) in each column is marked violett (resp. yellow).

In [ ]:
# All numeric columns excluding IDs and classifications
exclude_cols = ['string', 'counter_d', 'time_d'] + [col for col in nwp_geo.columns if col.startswith('y_')]
exclude_cols.extend(['area1', 'obj_area1', 'latcent1', 'loncent1', 'theta1', 'ar1', 'area2', 'obj_area2', 'latcent2', 'loncent2', 'theta2', 'ar2'])
numeric_cols = nwp_all.select_dtypes(include='number').columns
cols_to_summarize = [col for col in numeric_cols if (col not in exclude_cols and col not in nwp_geo.columns)]

merged = nwp_geo.merge(nwp_all[cols_to_summarize+['string']], on='string', how='inner')
summary_dict = {}

for model in y_names:
    grouped_all = merged.groupby(model)[cols_to_summarize].agg(['min', 'max', 'mean'])
    grouped_geo = nwp_geo.groupby(model)[nwp_geo.select_dtypes(include='number').columns[3:]].agg(['min', 'max', 'mean'])
    summary_dict[model] = [grouped_all, grouped_geo]

pd.set_option('display.precision', 2)
for model, df in summary_dict.items():
    print(f"\n Summary statistics from remaining data for {model}:")
    all = df[0]
    train_features = df[1]
    display(all.style.format("{:.2f}").background_gradient(axis=0, cmap ="viridis"))
    print(f"\n Summary statistics from data used to train {model}:")
    display(train_features.style.format("{:.2f}").background_gradient(axis=0, cmap ="viridis"))

In [ ]:
import pandas as pd
summary_y_ar_latcent_all = pd.read_csv('summary_y_ar_latcent_all.csv', sep = ',')
summary_y_ar_latcent_all

In [ ]:
import pandas as pd
summary_y_ar_latcent_area_edge_all = pd.read_csv('summary_y_ar_latcent_area_edge_all.csv', sep = ',')
summary_y_ar_latcent_area_edge_all

In [ ]:
import pandas as pd
summary_y_ar_latcent_area_edge_train = pd.read_csv('summary_y_ar_latcent_area_edge_train.csv', sep = ',')
summary_y_ar_latcent_area_edge_train

In [ ]:
import pandas as pd
summary_y_ar_latcent_scArea_all = pd.read_csv('summary_y_ar_latcent_scArea_all.csv', sep = ',')
summary_y_ar_latcent_scArea_all

In [ ]:
import pandas as pd
summary_y_ar_latcent_scArea_train = pd.read_csv('summary_y_ar_latcent_scArea_train.csv', sep = ',')
summary_y_ar_latcent_scArea_train

In [ ]:
import pandas as pd
summary_y_ar_latcent_train = pd.read_csv('summary_y_ar_latcent_train.csv', sep = ',')
summary_y_ar_latcent_train

In [ ]:
import pandas as pd
summary_y_ar_latcent_u_all = pd.read_csv('summary_y_ar_latcent_u_all.csv', sep = ',')
summary_y_ar_latcent_u_all

In [ ]:
import pandas as pd
summary_y_ar_latcent_u_train = pd.read_csv('summary_y_ar_latcent_u_train.csv', sep = ',')
summary_y_ar_latcent_u_train

### Schematic plot of vortex of representative sample for each found cluster
To visually characterize the dynamical structure of each cluster, a representative vortex configuration is selected. For each class in each model:

1. The cluster mean is computed for a set of geometric and diagnostic features (ar, latcent, area, etc.)
2. The sample closest to this mean in Euclidean space is selected as the representative day.
3. To illustrate the vortex evolution, the day **before** and **after** the representative are included in the plot.

Each 3-day window is visualized using the plot_polar_stereo function in overlay mode. If a split event or major warming is present in the window, this is noted in the output. This allows for qualitative assessment of how each class relates to key vortex phenomena and helps confirm the physical consistency of the clustering results.

In [ ]:
from sklearn.metrics.pairwise import euclidean_distances
from datetime import timedelta

foi = ['scaled_area', 'scaled_ar', 'scaled_latcent', 'scaled_kurtosis',
       'ssa_filtered_u']
for y in y_names:
    print('y: ', y)
    for c in nwp_geo[y].unique():
        print('class: ', c)
        # compute all means
        mean_vec = nwp_geo[nwp_geo[y] == c][foi].mean()
        # compute distance of each sample to mean
        dist_mean = euclidean_distances(nwp_geo[foi], [mean_vec])
        # determine index of sample closest to mean
        mean_idx = dist_mean.argmin()
        # obtain representative sample
        rep_mean = nwp_geo.iloc[mean_idx]
        print("Representative mean has class: ", rep_mean[y])

        dates = [rep_mean.string + timedelta(days=i) for i in range(0, 1)]
        samples = nwp_all[nwp_all.string.isin(dates)]

        print("Plotted vortices have classes: ", nwp_geo[nwp_geo.string.isin(dates)][y].tolist())

        if samples.form.sum() > 0:
            print('split event in time range')
        if samples.MW.sum() > 0:
            print('MW event in time range')

                # print(isinstance(samples, pd.DataFrame))
        viz.plot_polar_stereo(samples,
                              mode='single',
                              time_col='string',
                              filled=False,
                              figsize=(10, 10),
                              savefig=f"../output/nwp/rep_{y}_{rep_mean[y]}_nwp.png")

## Exploration of `ward` linkage

This part is for exploration and demonstration of the influence of parameter choice in models. Its interpretation is not done in detail and remains to be discussed.

In [ ]:
model = AgglomerativeClustering(linkage='ward')
k_max=10
gap_ward_ar_latcent = vortexclust.gap_statistic(nwp_geo[['scaled_ar', 'scaled_latcent']], k_max, 15, model=model)
gap_ward_ar_latcent_u = vortexclust.gap_statistic(nwp_geo[['scaled_ar', 'scaled_latcent', 'ssa_filtered_u']], k_max, 15, model=model)
gap_ward_ar_latcent_scArea = vortexclust.gap_statistic(nwp_geo[['scaled_ar', 'scaled_latcent', 'scaled_area']], k_max, 15, model=model)
gap_ward_ar_latcent_area_edge = vortexclust.gap_statistic(nwp_geo[['scaled_ar', 'scaled_latcent', 'scaled_area', 'ssa_filtered_edge']], k_max, 15, model=model)

In [ ]:
sil_ward_ar_latcent = vortexclust.silhouette_method(nwp_geo[['scaled_ar', 'scaled_latcent']], k_max, model=model)
sil_ward_ar_latcent_u = vortexclust.silhouette_method(nwp_geo[['scaled_ar', 'scaled_latcent', 'ssa_filtered_u']], k_max, model=model)
sil_ward_ar_latcent_scArea = vortexclust.silhouette_method(nwp_geo[['scaled_ar', 'scaled_latcent', 'scaled_area']], k_max, model=model)
sil_ward_ar_latcent_area_edge = vortexclust.silhouette_method(nwp_geo[['scaled_ar', 'scaled_latcent', 'scaled_area', 'ssa_filtered_u']], k_max, model=model)

In [ ]:
elbow_ward_ar_latcent = vortexclust.elbow_method(nwp_geo[['scaled_ar', 'scaled_latcent']], k_max, model=model)
elbow_ward_ar_latcent_u = vortexclust.elbow_method(nwp_geo[['scaled_ar', 'scaled_latcent', 'ssa_filtered_u']], k_max, model=model)
elbow_ward_ar_latcent_scArea = vortexclust.elbow_method(nwp_geo[['scaled_ar', 'scaled_latcent', 'scaled_area']], k_max, model=model)
elbow_ward_ar_latcent_area_edge = vortexclust.elbow_method(nwp_geo[['scaled_ar', 'scaled_latcent', 'scaled_area', 'ssa_filtered_u']], k_max, model=model)

In [ ]:
fig, ax = plt.subplots(2,2, figsize=(15, 10), sharex='all')
ax[0][0].errorbar(np.arange(1,k_max+1), gap_ward_ar_latcent[:, 0], yerr=gap_ward_ar_latcent[:, 1], label='AR, Latcent')
ax[0][0].errorbar(np.arange(1,k_max+1), gap_ward_ar_latcent_scArea[:, 0], yerr=gap_ward_ar_latcent_scArea[:, 1], label='AR, Latcent, Area')
ax[0][0].errorbar(np.arange(1,k_max+1), gap_ward_ar_latcent_u[:, 0], yerr=gap_ward_ar_latcent_u[:, 1], label='Ar, Latcent, Filtered wind speed')
ax[0][0].errorbar(np.arange(1,k_max+1), gap_ward_ar_latcent_area_edge[:, 0], yerr=gap_ward_ar_latcent_area_edge[:, 1], label='Ar, Latcent, Area, Edge')
ax[0][0].set_title('Gap statistic')
ax[0][0].legend(loc='upper right')

ax[1][0].plot(np.arange(1,k_max+1), elbow_ward_ar_latcent[0], label='AR, Latcent')
ax[1][0].plot(np.arange(1,k_max+1), elbow_ward_ar_latcent_scArea[0], label='AR, Latcent, Area')
ax[1][0].plot(np.arange(1,k_max+1), elbow_ward_ar_latcent_u[0], label='AR, Latcent, Filtered wind speed')
ax[1][0].plot(np.arange(1,k_max+1), elbow_ward_ar_latcent_area_edge[0], label='AR, Latcent, Area, Edge')
ax[1][0].set_title('Elbow method - Distortion')
ax[1][0].legend(loc='upper right')

ax[1][1].plot(np.arange(1,k_max+1), elbow_ward_ar_latcent[1], label='AR, Latcent')
ax[1][1].plot(np.arange(1,k_max+1), elbow_ward_ar_latcent_scArea[1], label='AR, Latcent, Area')
ax[1][1].plot(np.arange(1,k_max+1), elbow_ward_ar_latcent_u[1], label='AR, Latcent, Filtered wind speed')
ax[1][1].plot(np.arange(1,k_max+1), elbow_ward_ar_latcent_area_edge[1], label='AR, Latcent, Area, Edge')
ax[1][1].set_title('Elbow method - Inertia')
ax[1][1].legend(loc='upper right')

ax[0][1].plot(np.arange(2,k_max+1), sil_ward_ar_latcent, label='AR, Latcent')
ax[0][1].plot(np.arange(2,k_max+1), sil_ward_ar_latcent_scArea, label='AR, Latcent, Area')
ax[0][1].plot(np.arange(2,k_max+1), sil_ward_ar_latcent_u, label='AR, Latcent, Filtered wind speed')
ax[0][1].plot(np.arange(2,k_max+1), sil_ward_ar_latcent_area_edge, label='AR, Latcent, Area, Edge')
ax[0][1].set_title('Silhouette method')
ax[0][1].legend(loc='upper right')

plt.suptitle('Different k determination methods (NWP)')
plt.xticks(np.arange(1, k_max+1))
fig.text(0.5, 0, 'Number of clusters', ha='center')
plt.tight_layout()
plt.savefig("../output/nwp/Kopt_nwp_ward.png")
plt.show()

In [ ]:
print("Optimal number of clusters by each method: ")
p_1, p_2, p_3, p_4 = 1,1,1,1
for k in range(1, 10):
    if p_1 and (gap_ward_ar_latcent[k][0] >= gap_ward_ar_latcent[k+1][0] - gap_ward_ar_latcent[k+1][1]):
        print('Gap statistic (AR, Latcent): ', k+1) # index starts at 0, k starts at 1
        p_1=0
    if p_2 and (gap_ward_ar_latcent_scArea[k][0] >= gap_ward_ar_latcent_scArea[k+1][0] - gap_ward_ar_latcent_scArea[k+1][1]):
        print("Gap statistic (AR, Latcent, Area): ", k+1)
        p_2 = 0
    if p_4 and (gap_ward_ar_latcent_area_edge[k][0] >= gap_ward_ar_latcent_area_edge[k+1][0] - gap_ward_ar_latcent_area_edge[k+1][1]):
        print("Gap statistic (AR, Latcent, Area, Edge): ", k+1)
        p_4 = 0
    if p_3 and (gap_ward_ar_latcent_u[k][0] >= gap_ward_ar_latcent_u[k+1][0] - gap_ward_ar_latcent_u[k+1][1]):
        print("Gap statistic (AR, Latcent, filtered wind speed): ", k+1)
        p_3=0

print("Silhouette method (AR, Latcent): ", pd.DataFrame(sil_ward_ar_latcent).idxmax()[0]+2)
print("Silhouette method (AR, latcent, Area): ", pd.DataFrame(sil_ward_ar_latcent_scArea).idxmax()[0]+2)
print("Silhouette method (AR, latcent, Area, Edge): ", pd.DataFrame(sil_ward_ar_latcent_area_edge).idxmax()[0]+2)
print("Silhouette method (AR, latcent, filtered wind speed): ", pd.DataFrame(sil_ward_ar_latcent_u).idxmax()[0]+2)

## Optimal k for ward linkage

| Feature set                            | gap statistic | silhouette method | elbow (distortion) | elbow (inertia) | $k_{opt}$ |
|----------------------------------------|---------------|-------------------|--------------------|---------------|-----------|
| ar, latcent                            | 1 or 4        | 2 or 4            | 4                  | 4             | 4         |
| ar, latcent, scaled area               | 1, 2 or 4     | 2 or 4            | 2 or 4             | 2 or 4        | 4         |
| ar, latcent, filtered wind speed       | 1 or 3        | 3                 | 3                  | 3             | 3         |
| ar, latcent, scaled area, filtered edge| 1 or 3        | 4                 | 4 or 5             | 4 or 5        | 4         |

In [ ]:
features_kopt = [{'features' : ['scaled_ar', 'scaled_latcent'], 'k_opt' : 4, 'line':50},
                 {'features' : ['scaled_ar', 'scaled_latcent', 'scaled_area'], 'k_opt' : 4, 'line':60},
                 {'features' : ['scaled_ar', 'scaled_latcent', 'scaled_area', 'ssa_filtered_edge'], 'k_opt' : 4, 'line':65},
                 {'features' : ['scaled_ar', 'scaled_latcent', 'ssa_filtered_u'], 'k_opt' : 3, 'line':60}]
Y = []
base_model = AgglomerativeClustering(linkage='ward', compute_distances=True)

fig = plt.figure(figsize=(15,10))
for idx, feat_k in enumerate(features_kopt):
    model = clone(base_model)
    model.set_params(n_clusters = feat_k['k_opt'])
    model.fit(nwp_geo[feat_k['features']])
    ax = fig.add_subplot(2,2,idx+1)
    ax.set_title(str(feat_k['features'])[1:-1]+", k="+str(feat_k['k_opt']))
    viz.plot_dendrogram(model, truncate_mode='level', p=4, direction='LR')
    ax.axvline(feat_k['line'], ls='--', color='black')
    y_pred = model.labels_.astype(int)
    Y.append(y_pred)
plt.tight_layout()
plt.savefig("../output/nwp/dendrogram_ward.png")
plt.show()

fig = plt.figure(figsize=(15,10))
for idx, feat_k in enumerate(features_kopt):
    if len(feat_k['features']) == 3:
        ax = fig.add_subplot(2,2,idx+1, projection='3d')
        ax.scatter(nwp_geo[feat_k['features']].iloc[:, 0],
                   nwp_geo[feat_k['features']].iloc[:, 1],
                   nwp_geo[feat_k['features']].iloc[:, 2],
                   c=Y[idx], cmap='tab10')
        ax.set_facecolor((0, 0, 0, 0))
        ax.set_xlabel(feat_k['features'][0])
        ax.set_ylabel(feat_k['features'][1])
        ax.set_zlabel(feat_k['features'][2])
    else:
        ax = fig.add_subplot(2,2, idx+1)
        ax.scatter(nwp_geo[feat_k['features']].iloc[:, 0],
                   nwp_geo[feat_k['features']].iloc[:, 1],
                   c=Y[idx], cmap='tab10')
        ax.set_xlabel(feat_k['features'][0])
        ax.set_ylabel(feat_k['features'][1])
    plt.subplots_adjust(hspace=0.4, wspace=0.4)
    ax.set_title(str(feat_k['features'])[1:-1]+", k="+str(feat_k['k_opt']))
plt.show()

In [ ]:
y_names_ward = ['y_ward_ar_latcent', 'y_ward_ar_latcent_scArea', 'y_ward_ar_latcent_area_edge', 'y_ward_ar_latcent_u']
nwp_geo[y_names_ward] = pd.DataFrame(Y).T
# nwp_geo.to_csv("../data/nwp_geo.csv")

# nwp_geo['y_ward_ar_latcent'] = nwp_geo['y_ward_ar_latcent'].replace({0: 'D', 1:'DS', 2:'S', 3: 'D'})
print("Averages per class and features:")
print(nwp_geo[['y_ward_ar_latcent', 'scaled_ar', 'scaled_latcent']].groupby(['y_ward_ar_latcent']).mean())
print(nwp_geo['y_ward_ar_latcent'].value_counts())
plot_hist_per_class(nwp_geo, # data
                    features_kopt[0], # information about used feature and k_opt
                    'y_ward_ar_latcent',
                    savefig="../output/hist_nwp_ward_mBasic_") # column name with y values

In [ ]:
print("Averages per class and features:")
nwp_geo['y_ward_ar_latcent_u'] = nwp_geo['y_ward_ar_latcent_u'].replace({0: 'D', 1:'U', 2:'S'})

print(nwp_geo[['y_ward_ar_latcent_u', 'scaled_ar', 'scaled_latcent', 'ssa_filtered_u']].groupby(['y_ward_ar_latcent_u']).mean())
print(nwp_geo['y_ward_ar_latcent_u'].value_counts())
plot_hist_per_class(nwp_geo, # data
                    features_kopt[3], # information about used feature and k_opt
                    'y_ward_ar_latcent_u',
                    savefig="../output/nwp/hist_nwp_ward_mWind_") # column name with y values

In [ ]:
print("Averages per class and features:")
nwp_geo['y_ward_ar_latcent_scArea'] = nwp_geo['y_ward_ar_latcent_scArea'].replace({0: 'D', 1:'S', 2:'U', 3: 'L'})

print(nwp_geo[['y_ward_ar_latcent_scArea', 'scaled_ar', 'scaled_latcent', 'ssa_filtered_u']].groupby(['y_ward_ar_latcent_scArea']).mean())
print(nwp_geo['y_ward_ar_latcent_scArea'].value_counts())
plot_hist_per_class(nwp_geo, # data
                    features_kopt[1], # information about used feature and k_opt
                    'y_ward_ar_latcent_scArea',
                    savefig="../output/nwp/hist_nwp_ward_mscArea_") # column name with y values

In [ ]:
print("Averages per class and features:")
# nwp_geo['y_ward_ar_latcent_area_edge'] = nwp_geo['y_ward_ar_latcent_area_edge'].replace({3:'L'})

print(nwp_geo[['y_ward_ar_latcent_area_edge', 'scaled_ar', 'scaled_latcent', 'ssa_filtered_edge', 'scaled_area']].groupby(['y_ward_ar_latcent_area_edge']).mean())
print(nwp_geo['y_ward_ar_latcent_area_edge'].value_counts())
plot_hist_per_class(nwp_geo, # data
                    features_kopt[2], # information about used feature and k_opt
                    'y_ward_ar_latcent_area_edge',
                    savefig="../output/nwp/hist_nwp_ward_medge_") # column name with y values

# Appendix
For reasons of interest in the data the following was kept.

## Computation of the partial autocorrelation function
The partial autocorrelation function does not exhibit any remarkable perks except for the wind speed reduced to DJFM. The visible peaks correspond closely to the suggested time window of 120 days and indicate a 120d periodic signal in the data.

In [ ]:
# scaling of all data
sc = StandardScaler()
nwp_all[['scaled_area', 'scaled_ar', 'scaled_latcent', 'scaled_u']] = sc.fit_transform(nwp_all[['area', 'ar', 'latcent', 'u']])

In [ ]:
from statsmodels.graphics.tsaplots import plot_pacf

fig, ax = plt.subplots(2,2, figsize=(15, 10))
ax = ax.flatten()
plot_pacf(nwp_geo['scaled_area'], lags=500, ax=ax[0], label='PACF scaled area (DJFM)', marker=None)
plot_pacf(nwp_all['scaled_area'], lags=500, ax=ax[1], label='PACF scaled area (entire timeseries)', marker=None)
plot_pacf(nwp_geo['scaled_u'], lags=500, ax=ax[2], label='PACF wind speed (DJFM)', marker=None)
plot_pacf(nwp_all['scaled_u'], lags=500, ax=ax[3], label='PACF wind speed (entire timeseries)', marker=None)

ax[0].set_title('Partial autocorrelation of area (DJFM)')
ax[1].set_title('Partial autocorrelation of area (entire timeseries)')
ax[2].set_title('Partial autocorrelation of wind speed (DJFM)')
ax[3].set_title('Partial autocorrelation of wind speed (entire timeseries)')

for i in range(2):
    ax[i].axhline(y=0.05, linestyle='--', color='black', linewidth=1)
    ax[i].axhline(y=-0.05, linestyle='--', color='black', linewidth=1)
    ax[i].set_xlim(0, 500)
    ax[i].set_xticks(positions[:5])
    for x in positions:
        ax[i].axvline(x=x, color='grey', linestyle=':', linewidth=0.8)
plt.tight_layout()
plt.show()

## Year over Year averages

Below the yearly seasonality is computed with the year over year average as well as the reconstruction of the yearly seasonality with SSA.

In [ ]:
from vortexclust.workflows.demo import plot_ssa_grid
# Average data
avg_year = nwp_all.groupby('year').mean()
avg_year.index = pd.to_datetime(avg_year.index, format='%Y')

avg_day_over_year = nwp_all.groupby(['month', 'day']).mean()
avg_day_over_year.index = pd.to_datetime({"year": 2000,  # just a sample year, not displayed in plot
                                          "month": avg_day_over_year.index.get_level_values(0),
                                          "day": avg_day_over_year.index.get_level_values(1)})
# compute SSA
ssa_long_term = SingularSpectrumAnalysis(window_size=10)
ssa_yearly_cycle = SingularSpectrumAnalysis(window_size=90)

long_term_area = ssa_long_term.fit_transform(avg_year['area'].values.reshape(1, -1))
long_term_ar = ssa_long_term.fit_transform(avg_year['ar'].values.reshape(1, -1))
long_term_latcent = ssa_long_term.fit_transform(avg_year['latcent'].values.reshape(1, -1))

year_seasonality_area = ssa_yearly_cycle.fit_transform(avg_day_over_year['area'].values.reshape(1, -1))
year_seasonality_ar = ssa_yearly_cycle.fit_transform(avg_day_over_year['ar'].values.reshape(1, -1))
year_seasonality_latcent = ssa_yearly_cycle.fit_transform(avg_day_over_year['latcent'].values.reshape(1, -1))

# Plot SSA
plot_ssa_grid(
    data_series=[avg_year, avg_day_over_year],
    ssa_results=[[long_term_area, long_term_ar, long_term_latcent],
                 [year_seasonality_area, year_seasonality_ar, year_seasonality_latcent]],
    index_format=['%Y', '%b %d'],
    labels=['area', 'ar', 'latcent'],
    titles=['long term trend of average per year', 'yearly cycle of average per day over all years'],
    used_signals=4
)

## Abbreviations

|         |                                                |
|--------:|:-----------------------------------------------|
|     H11 | Hannachi et al. 2011                           |
|     IAP | Institute for Atmospheric Physics Kühlungsborn |
|    DJFM | December, January, February, March             |
|      ar | Aspect ratio                                   |
| latcent | centroid latitude                              |
|     SSW | Sudden Stratospheric Warming                   |
|       S | Cluster of split vortices                      |
|       D | Cluster of displaced vortices                  |
|       L | Cluster of large vortices                      |
|       U | Cluster of undisturbed vortices                |
|      E  | Cluster of extreme samples                     |
|     ACF | Autocorrelation function                       |
|    PACF | Partial Autocorrelation function               |

## Abbreviations

|         |                                                |
|--------:|:-----------------------------------------------|
|     H11 | Hannachi et al. 2011                           |
|     IAP | Institute for Atmospheric Physics Kühlungsborn |
|    DJFM | December, January, February, March             |
|      ar | Aspect ratio                                   |
| latcent | centroid latitude                              |
|     SSW | Sudden Stratospheric Warming                   |
|       S | Cluster of split vortices                      |
|       D | Cluster of displaced vortices                  |
|       L | Cluster of large vortices                      |
|       U | Cluster of undisturbed vortices                |
|      E  | Cluster of extreme samples                     |
|     ACF | Autocorrelation function                       |
|    PACF | Partial Autocorrelation function               |